# Classify Movie Reviews as Positive or Negative using Bag of Words

**Aim:** To classify movie reviews as positive or negative using a Bag of Words (BoW) representation and a
machine-learning classifier.

**Dataset:** IMDb movie reviews (25,000 training and 25,000 test reviews, balanced).

**Steps:** Bag of Words features → train Logistic Regression → evaluate (accuracy, precision, recall, F1) →
Naive Bayes and SVM for comparison → **test** (model comparison, cross-validation, ROC curves, custom reviews,
error analysis).

In [ ]:
!pip install -q datasets

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from datasets import load_dataset
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    roc_curve,
    classification_report,
    confusion_matrix
)

## 1. Load and explore the dataset

In [ ]:
# [LOAD]
dataset = load_dataset("stanfordnlp/imdb")

# Convert to Pandas
train_df = pd.DataFrame(dataset["train"])
test_df = pd.DataFrame(dataset["test"])

print("Training samples:", len(train_df))
print("Testing samples :", len(test_df))
display(train_df.head())

In [ ]:
print("Training dataset shape:", train_df.shape)
print("Testing dataset shape :", test_df.shape)

print("\nLabels (0 = negative, 1 = positive):")
print(train_df["label"].value_counts())

print("\nExample review:")
print(train_df["text"].iloc[0][:600], "...")
print("\nLabel:", train_df["label"].iloc[0])

In [ ]:
# Review length distribution
lengths = train_df["text"].str.split().str.len()
print(lengths.describe().round(1))

plt.figure(figsize=(8, 4))
plt.hist(lengths, bins=50)
plt.xlabel("Words per review")
plt.ylabel("Number of reviews")
plt.title("Review length distribution (training set)")
plt.show()

In [ ]:
X_train = train_df["text"]
y_train = train_df["label"]

X_test = test_df["text"]
y_test = test_df["label"]

print("Training reviews:", len(X_train))
print("Testing reviews :", len(X_test))

## 2. Bag of Words feature extraction

In [ ]:
vectorizer = CountVectorizer(
    lowercase=True,
    stop_words="english"
)

# Learn vocabulary and transform training reviews
X_train_bow = vectorizer.fit_transform(X_train)

# Transform testing reviews using the same vocabulary
X_test_bow = vectorizer.transform(X_test)

print("Training BoW shape:", X_train_bow.shape)
print("Testing BoW shape :", X_test_bow.shape)

In [ ]:
# Get vocabulary
vocabulary = vectorizer.get_feature_names_out()

print("Number of features:", len(vocabulary))
print("\nFirst 30 vocabulary words:")
print(vocabulary[:30])

In [ ]:
sample_review = X_train.iloc[0]
sample_vector = vectorizer.transform([sample_review])

print("Review:")
print(sample_review[:300], "...")

print("\nNon-zero BoW features:")
feature_indices = sample_vector.nonzero()[1]

for index in feature_indices[:20]:
    print(vocabulary[index], "→", sample_vector[0, index])

## 3. Train a classifier (Logistic Regression)

In [ ]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train_bow, y_train)

print("Model training completed.")

In [ ]:
y_pred = model.predict(X_test_bow)

print("Predictions generated successfully.")
print("\nFirst 20 predictions:")
print(y_pred[:20])
print("\nActual labels:")
print(y_test.iloc[:20].values)

## 4. Evaluate: accuracy, precision, recall, F1

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("========== MODEL PERFORMANCE ==========")
print(f"Accuracy  : {accuracy:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1-score  : {f1:.4f}")

print("\n========== PERFORMANCE (%) ==========")
print(f"Accuracy  : {accuracy * 100:.2f}%")
print(f"Precision : {precision * 100:.2f}%")
print(f"Recall    : {recall * 100:.2f}%")
print(f"F1-score  : {f1 * 100:.2f}%")

In [ ]:
print(classification_report(y_test, y_pred, target_names=["Negative", "Positive"]))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
print(cm)

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    xticklabels=["Negative", "Positive"],
    yticklabels=["Negative", "Positive"]
)
plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")
plt.title("Confusion Matrix")
plt.show()

## 5. Predict new reviews

In [ ]:
new_reviews = [
    "This movie was absolutely fantastic and I loved every minute of it.",
    "The movie was boring, poorly written and a complete waste of time.",
    "Amazing acting and an excellent storyline.",
    "I hated this movie. It was terrible and disappointing."
]

# Convert reviews to BoW
new_reviews_bow = vectorizer.transform(new_reviews)

# Predict
predictions = model.predict(new_reviews_bow)

print("========== NEW REVIEW PREDICTIONS ==========\n")
for review, prediction in zip(new_reviews, predictions):
    sentiment = "Positive" if prediction == 1 else "Negative"
    print("Review    :", review)
    print("Prediction:", sentiment)
    print()

## 6. Other classifiers: Naive Bayes and SVM

In [ ]:
# Naive Bayes
nb_model = MultinomialNB()
nb_model.fit(X_train_bow, y_train)
nb_predictions = nb_model.predict(X_test_bow)

print("Naive Bayes Performance")
print("------------------------")
print("Accuracy :", accuracy_score(y_test, nb_predictions))
print("Precision:", precision_score(y_test, nb_predictions))
print("Recall   :", recall_score(y_test, nb_predictions))
print("F1-score :", f1_score(y_test, nb_predictions))

In [ ]:
# Support Vector Machine (linear)
svm_model = LinearSVC(max_iter=5000)
svm_model.fit(X_train_bow, y_train)
svm_predictions = svm_model.predict(X_test_bow)

print("SVM Performance")
print("------------------------")
print("Accuracy :", accuracy_score(y_test, svm_predictions))
print("Precision:", precision_score(y_test, svm_predictions))
print("Recall   :", recall_score(y_test, svm_predictions))
print("F1-score :", f1_score(y_test, svm_predictions))

---
# Testing the classifiers

1. **Model comparison on the 25,000 unseen test reviews** - accuracy, precision, recall, F1 and ROC-AUC.
2. **Cross-validation** - is the result stable or just lucky?
3. **ROC curves.**
4. **Custom review test** - hand-written reviews with a known correct answer, including difficult cases.
5. **Most informative words** and **error analysis**.

## Test 1: compare the three models on the test set

In [ ]:
scores = {
    "Logistic Regression": model.predict_proba(X_test_bow)[:, 1],
    "Naive Bayes":         nb_model.predict_proba(X_test_bow)[:, 1],
    "SVM (linear)":        svm_model.decision_function(X_test_bow),
}
predictions_by_model = {
    "Logistic Regression": y_pred,
    "Naive Bayes":         nb_predictions,
    "SVM (linear)":        svm_predictions,
}

rows = []
for name, pred in predictions_by_model.items():
    rows.append({
        "Model": name,
        "Accuracy":  round(accuracy_score(y_test, pred), 4),
        "Precision": round(precision_score(y_test, pred), 4),
        "Recall":    round(recall_score(y_test, pred), 4),
        "F1-score":  round(f1_score(y_test, pred), 4),
        "ROC-AUC":   round(roc_auc_score(y_test, scores[name]), 4),
    })

comparison = pd.DataFrame(rows).set_index("Model")
display(comparison)
print("Best model by F1-score:", comparison["F1-score"].idxmax())

comparison[["Accuracy", "Precision", "Recall", "F1-score"]].plot(kind="bar", figsize=(10, 5), ylim=(0.8, 0.9))
plt.title("Model comparison on the IMDb test set")
plt.ylabel("Score")
plt.xticks(rotation=0)
plt.legend(loc="lower right")
plt.show()

In [ ]:
# Confusion matrices of all three models side by side
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, pred) in zip(axes, predictions_by_model.items()):
    sns.heatmap(confusion_matrix(y_test, pred), annot=True, fmt="d", cbar=False,
                xticklabels=["Neg", "Pos"], yticklabels=["Neg", "Pos"], ax=ax)
    ax.set_title(name)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
plt.tight_layout()
plt.show()

## Test 2: 5-fold cross-validation

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_score, StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_rows = []
for name, clf in [
    ("Logistic Regression", LogisticRegression(max_iter=1000)),
    ("Naive Bayes", MultinomialNB()),
    ("SVM (linear)", LinearSVC(max_iter=5000)),
]:
    pipe = make_pipeline(CountVectorizer(lowercase=True, stop_words="english"), clf)
    fold_scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="accuracy")
    cv_rows.append({
        "Model": name,
        "Fold accuracies": ", ".join(f"{s:.3f}" for s in fold_scores),
        "Mean": round(fold_scores.mean(), 4),
        "Std dev": round(fold_scores.std(), 4),
    })

display(pd.DataFrame(cv_rows))
print("A small standard deviation means the accuracy does not depend on which reviews are used for training.")

## Test 3: ROC curves

In [ ]:
plt.figure(figsize=(7, 6))
for name, s in scores.items():
    fpr, tpr, _ = roc_curve(y_test, s)
    plt.plot(fpr, tpr, label=f"{name} (AUC = {roc_auc_score(y_test, s):.3f})")

plt.plot([0, 1], [0, 1], "k--", label="Random guess")
plt.xlabel("False positive rate")
plt.ylabel("True positive rate")
plt.title("ROC curves - movie review sentiment")
plt.legend(loc="lower right")
plt.show()

## Test 4: custom reviews with a known correct answer

In [ ]:
# (review, correct label)  1 = positive, 0 = negative
custom_tests = [
    ("An absolute masterpiece. Brilliant acting, a gripping story and a beautiful soundtrack.", 1),
    ("One of the best films I have seen this year, I would happily watch it again.", 1),
    ("Wonderful performances and a heartwarming ending. Highly recommended!", 1),
    ("A touching, funny and well made movie that the whole family will enjoy.", 1),
    ("Terrible. The plot made no sense and the acting was painfully bad.", 0),
    ("A boring, predictable mess. I wanted my money back.", 0),
    ("Awful script, wooden actors and an ending that was a complete waste of time.", 0),
    ("I fell asleep halfway through, dull and disappointing from start to finish.", 0),
    # difficult cases
    ("This movie was not good at all.", 0),
    ("I did not hate it, in fact it was surprisingly enjoyable.", 1),
    ("The first half is slow and boring but the ending is brilliant and moving.", 1),
    ("Great visuals, shame about the dreadful story and the terrible dialogue.", 0),
]

texts  = [t for t, _ in custom_tests]
truth  = np.array([l for _, l in custom_tests])
pred   = model.predict(vectorizer.transform(texts))
proba  = model.predict_proba(vectorizer.transform(texts))

custom_df = pd.DataFrame({
    "Review": [t[:70] + ("..." if len(t) > 70 else "") for t in texts],
    "Expected": ["Positive" if l else "Negative" for l in truth],
    "Predicted": ["Positive" if p else "Negative" for p in pred],
    "Confidence": [f"{max(p) * 100:.1f}%" for p in proba],
    "Correct": ["yes" if a == b else "NO" for a, b in zip(truth, pred)],
})
display(custom_df)

easy = slice(0, 8)
print(f"Accuracy on the 8 clear reviews     : {(truth[easy] == pred[easy]).mean() * 100:.1f}%")
print(f"Accuracy on the 4 difficult reviews : {(truth[8:] == pred[8:]).mean() * 100:.1f}%")
print(f"Accuracy on all 12 custom reviews   : {(truth == pred).mean() * 100:.1f}%")

In [ ]:
def predict_sentiment(review):
    """Predict the sentiment of one review and show the confidence."""
    p = model.predict_proba(vectorizer.transform([review]))[0]
    label = "Positive" if p[1] >= 0.5 else "Negative"
    print(f"{label}  (positive probability = {p[1] * 100:.1f}%)  ->  {review}")

predict_sentiment("What a wonderful, charming film!")
predict_sentiment("Dull, slow and badly acted.")

## Test 5: most informative words

In [ ]:
coefficients = model.coef_[0]
top_pos = np.argsort(coefficients)[-15:]
top_neg = np.argsort(coefficients)[:15]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].barh(vocabulary[top_pos], coefficients[top_pos], color="green")
axes[0].set_title("Words that push towards POSITIVE")
axes[1].barh(vocabulary[top_neg], coefficients[top_neg], color="red")
axes[1].set_title("Words that push towards NEGATIVE")
plt.tight_layout()
plt.show()

## Test 6: error analysis - reviews the model gets wrong

In [ ]:
wrong = np.where(y_pred != y_test.values)[0]
print(f"Misclassified reviews: {len(wrong)} of {len(y_test)} ({len(wrong) / len(y_test) * 100:.1f}%)")

proba_test = model.predict_proba(X_test_bow)
# the most confident mistakes show what the model misunderstands
confident_wrong = wrong[np.argsort(-np.abs(proba_test[wrong, 1] - 0.5))[:5]]

for i in confident_wrong:
    actual = "Positive" if y_test.iloc[i] == 1 else "Negative"
    pred_label = "Positive" if y_pred[i] == 1 else "Negative"
    print("-" * 80)
    print(f"Actual: {actual}   Predicted: {pred_label}   (positive probability {proba_test[i, 1]:.2f})")
    print(X_test.iloc[i][:350], "...")

## Test 7: effect of removing negation words

scikit-learn's English stop-word list contains **not**, **no**, **never** and **nor**. Removing them makes
"not good" look like "good". Here the same Logistic Regression is trained with those words kept.

In [ ]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

negations = {"not", "no", "nor", "never", "cannot", "nothing", "none"}
stop_keep_negation = sorted(ENGLISH_STOP_WORDS - negations)

vec_neg = CountVectorizer(lowercase=True, stop_words=stop_keep_negation, ngram_range=(1, 2), min_df=2)
Xtr_neg = vec_neg.fit_transform(X_train)
Xte_neg = vec_neg.transform(X_test)

model_neg = LogisticRegression(max_iter=1000)
model_neg.fit(Xtr_neg, y_train)
pred_neg = model_neg.predict(Xte_neg)

pred_custom_neg = model_neg.predict(vec_neg.transform(texts))

ablation = pd.DataFrame({
    "Variant": ["Original (stop words removed, unigrams)", "Keep negations + bigrams"],
    "Test accuracy": [round(accuracy_score(y_test, y_pred), 4), round(accuracy_score(y_test, pred_neg), 4)],
    "Test F1": [round(f1_score(y_test, y_pred), 4), round(f1_score(y_test, pred_neg), 4)],
    "Custom: difficult reviews": [f"{(truth[8:] == pred[8:]).sum()} / 4", f"{(truth[8:] == pred_custom_neg[8:]).sum()} / 4"],
})
display(ablation)

## Conclusion

Bag of Words features were extracted from IMDb reviews and a Logistic Regression classifier reached about 86 %
accuracy. Naive Bayes and an SVM were trained for comparison. The testing section compared the models on 25,000
unseen reviews, confirmed the result with 5-fold cross-validation and ROC curves, checked hand-written reviews
(including difficult negations), listed the most informative words and studied the model's mistakes.